# DSA 210 — Spotify Project
## Notebook 1: Data Collection & Preparation
**Selin Nardal**

This notebook loads the raw Spotify streaming history (extended format), cleans it, and prepares a final dataset for EDA and hypothesis testing.

In [1]:
import json
import pandas as pd
import numpy as np
import os

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 
1. Load Raw Data

The extended streaming history comes in two JSON files covering Oct 2023 – Apr 2026.
Each record contains timestamp, track name, artist, album, Spotify URI, playback duration, shuffle/skip flags, and platform.

In [2]:
# ── Adjust these paths if your data folder is different ──
DATA_FILES = [
    'data/Streaming_History_Audio_2023-2025_0.json',
    'data/Streaming_History_Audio_2025-2026_1.json',
]

records = []
for path in DATA_FILES:
    with open(path, encoding='utf-8') as f:
        chunk = json.load(f)
    records.extend(chunk)

df_raw = pd.DataFrame(records)
print(f'Total raw records : {len(df_raw):,}')
print(f'Columns           : {list(df_raw.columns)}')
df_raw.head(3)

Total raw records : 21,424
Columns           : ['ts', 'platform', 'ms_played', 'conn_country', 'ip_addr', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'episode_name', 'episode_show_name', 'spotify_episode_uri', 'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp', 'incognito_mode']


,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2023-10-16T15:08:11Z,osx,8549,TR,159.20.68.17,Die For You,The Weeknd,Starboy,spotify:track:2LBqCSwhJGcFQeTHMVGwy3,None,...,None,None,None,remote,endplay,False,True,False,NaN,False
1,2023-10-16T15:11:32Z,osx,200445,TR,159.20.68.17,The Color Violet,Tory Lanez,Alone At Prom,spotify:track:3azJifCSqg9fRij2yKIbWz,None,...,None,None,None,clickrow,fwdbtn,False,True,False,NaN,False
2,2023-10-16T15:11:46Z,osx,13296,TR,159.20.68.17,Poison,Brent Faiyaz,Poison,spotify:track:5NijSs5dAwaIybq1GaRTIe,None,...,None,None,None,fwdbtn,endplay,False,True,False,NaN,False


## 2. Filter Music Tracks Only

The raw file also contains podcast and audiobook rows (where `master_metadata_track_name` is null). We keep only music.

In [3]:
df_music = df_raw[df_raw['master_metadata_track_name'].notna()].copy()
print(f'Music records : {len(df_music):,}  ({len(df_raw)-len(df_music):,} non-music rows dropped)')

Music records : 21,387  (37 non-music rows dropped)


## 3. Rename & Type Conversions

In [4]:
df = df_music.rename(columns={
    'ts'                                  : 'timestamp',
    'ms_played'                           : 'ms_played',
    'master_metadata_track_name'          : 'track_name',
    'master_metadata_album_artist_name'   : 'artist',
    'master_metadata_album_album_name'    : 'album',
    'spotify_track_uri'                   : 'track_uri',
})

# Parse timestamp → datetime (UTC)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# Convert to Turkey local time (UTC+3)
df['timestamp_local'] = df['timestamp'].dt.tz_convert('Europe/Istanbul')

# Derived time features
df['date']        = df['timestamp_local'].dt.date
df['hour']        = df['timestamp_local'].dt.hour
df['day_of_week'] = df['timestamp_local'].dt.day_name()   # Monday … Sunday
df['month']       = df['timestamp_local'].dt.to_period('M').astype(str)
df['is_weekend']  = df['timestamp_local'].dt.dayofweek >= 5   # Sat=5, Sun=6

# Duration in seconds and minutes
df['sec_played'] = df['ms_played'] / 1000
df['min_played'] = df['sec_played'] / 60

# Time-of-day buckets
def time_bucket(hour):
    if 5 <= hour < 12:  return 'Morning'
    if 12 <= hour < 17: return 'Afternoon'
    if 17 <= hour < 21: return 'Evening'
    return 'Night'      # 21-04

df['time_of_day'] = df['hour'].apply(time_bucket)

# Hypothesis 1 specific: late-night (23:00-02:00) vs daytime (09:00-17:00)
df['h1_group'] = None
df.loc[df['hour'].isin([23, 0, 1, 2]), 'h1_group'] = 'late_night'
df.loc[(df['hour'] >= 9) & (df['hour'] < 17), 'h1_group'] = 'daytime'

print(df[['timestamp_local','track_name','artist','hour','day_of_week','is_weekend','time_of_day','h1_group']].head(5))

            timestamp_local        track_name        artist  hour day_of_week  \
0 2023-10-16 18:08:11+03:00       Die For You    The Weeknd    18      Monday   
1 2023-10-16 18:11:32+03:00  The Color Violet    Tory Lanez    18      Monday   
2 2023-10-16 18:11:46+03:00            Poison  Brent Faiyaz    18      Monday   
3 2023-10-16 18:11:48+03:00           Starboy    The Weeknd    18      Monday   
4 2023-10-16 18:12:25+03:00           Starboy    The Weeknd    18      Monday   

   is_weekend time_of_day h1_group  
0       False     Evening     None  
1       False     Evening     None  
2       False     Evening     None  
3       False     Evening     None  
4       False     Evening     None  


/var/folders/sf/s9_171315pq6m85wsr95h_380000gn/T/ipykernel_54174/3370317502.py:20: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month']       = df['timestamp_local'].dt.to_period('M').astype(str)


## 4. Academic Calendar Labels

Sabancı University exam periods (approximate) are labelled for RQ2.

In [5]:
# Define exam windows (add/adjust as needed)
EXAM_PERIODS = [
    ('2024-01-01', '2024-01-20'),   # Final exams — Fall 2023
    ('2024-04-15', '2024-05-10'),   # Final exams — Spring 2024
    ('2025-01-06', '2025-01-20'),   # Final exams — Fall 2024
    ('2025-04-14', '2025-05-10'),   # Final exams — Spring 2025
    ('2026-01-06', '2026-01-20'),   # Final exams — Fall 2025
]

df['is_exam'] = False
for start, end in EXAM_PERIODS:
    mask = (df['timestamp_local'].dt.date >= pd.Timestamp(start).date()) & \
           (df['timestamp_local'].dt.date <= pd.Timestamp(end).date())
    df.loc[mask, 'is_exam'] = True

print(f"Exam-period rows    : {df['is_exam'].sum():,}")
print(f"Non-exam rows       : {(~df['is_exam']).sum():,}")

Exam-period rows    : 2,972
Non-exam rows       : 18,415


## 
5. Remove Near-Zero Plays

Rows with < 10 seconds of play time are skips/errors and are excluded.

In [6]:
before = len(df)
df = df[df['sec_played'] >= 10].copy()
print(f'Dropped {before - len(df):,} near-zero rows. Remaining: {len(df):,}')

Dropped 5,806 near-zero rows. Remaining: 15,581


## 6. Save Clean Dataset

In [7]:
os.makedirs('data', exist_ok=True)
df.to_csv('data/streaming_clean.csv', index=False)
print(f'Saved data/streaming_clean.csv  ({len(df):,} rows)')

# Quick summary
print('\n── Dataset Summary ──')
print(f'Date range  : {df["timestamp_local"].min().date()} → {df["timestamp_local"].max().date()}')
print(f'Total plays : {len(df):,}')
print(f'Unique tracks  : {df["track_name"].nunique():,}')
print(f'Unique artists : {df["artist"].nunique():,}')
print(f'Total listening: {df["min_played"].sum()/60:.1f} hours')

Saved data/streaming_clean.csv  (15,581 rows)

── Dataset Summary ──
Date range  : 2023-10-16 → 2026-03-30
Total plays : 15,581
Unique tracks  : 5,272
Unique artists : 2,737
Total listening: 667.7 hours


## 7. (Optional) Enrich with Spotify API Audio Features

This step fetches `valence`, `energy`, `danceability`, `tempo` etc. via the Spotify Web API.
You need a `SPOTIPY_CLIENT_ID` and `SPOTIPY_CLIENT_SECRET` (free from developer.spotify.com).

> **Skip this cell if you don't have API keys yet — EDA and hypothesis tests work without it.**

In [11]:
RUN_API = Fals   # ← set to True when you have credentials

if RUN_API:
    import spotipy
    from spotipy.oauth2 import SpotifyClientCredentials

    sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
        client_id     = '293d4e4819f44f2ea89f71cd46468b57',
        client_secret = 'd28cd7bce3fb45ce8c7835e4fa0b0c1f',
    ))

    unique_uris = df['track_uri'].dropna().unique().tolist()
    track_ids   = [u.split(':')[-1] for u in unique_uris]

    features = []
    batch = 100
    for i in range(0, len(track_ids), batch):
        chunk = track_ids[i:i+batch]
        result = sp.audio_features(chunk)
        features.extend([r for r in result if r])
        if i % 1000 == 0:
            print(f'  Fetched {i}/{len(track_ids)}...')

    df_feat = pd.DataFrame(features)[['id','valence','energy','danceability','tempo','acousticness','instrumentalness','speechiness']]
    df_feat['track_uri'] = 'spotify:track:' + df_feat['id']

    df = df.merge(df_feat[['track_uri','valence','energy','danceability','tempo','acousticness']], on='track_uri', how='left')
    df.to_csv('data/streaming_with_features.csv', index=False)
    print(f'Saved streaming_with_features.csv with {df["valence"].notna().sum():,} tracks enriched.')
else:
    print('API step skipped. Using streaming_clean.csv for analysis.')

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=3azJifCSqg9fRij2yKIbWz,5NijSs5dAwaIybq1GaRTIe,7MXVkk9YMctZqd1Srtv4MB,65tIJClJx8fHo6YW4wVDhi,5xOAQ5o5mPI2FZZEOu1NZL,7x9aauaA9cu6tyfpHnqDLo,56y1jOTK0XSvJzVv9vHQBK,3rUGC1vUpkDG9CZFHMur1t,01qFKNWq73UfEslI0GvumE,2YSzYUF3jWqb9YP9VXmpjE,1BxfuPKGuaTgP7aM0Bbdwr,5RqSsdzTNPX1uzkmlHCFvK,3vkCueOmm7xQDoJ17W1Pm3,5mjYQaktjmjcMKcUIcqz4s,7ABLbnD53cQK00mhcaOUVG,4bw8mcDUSRWfQo63ZTYRnU,1odExI7RdWc4BT515LTAwj,6uIIdjYTxxpWOyWuVXrKQO,59NraMJsLaMCVtwXTSia8i,4Dvkj6JhhA12EX05fT7y2e,1Qrg8KqiBpW07V7PNxwwwL,3717eMglfGPYr9YKzdsiho,4hv9mEWi4911k1uhU4fPEH,3si1puqOHdvGAFZ7Z4VU27,5yy7ct80WZm20IrCh7KVb6,6hjbdWzbOX4OxBPZUxJVNH,11xjDJ7i9BLd1AYMedwJQK,5sfxl2zZcEdWQOF4ymxGpd,0iqfDCsvFZr9YBwRFqnai6,0ytdPqBPMEHhgXxLbx472A,3ZL94olNtSVGYR1Bmc9acW,3b8UW227uLkRDvd0w9e1Sw,6TQj22qKkIAd9cuBiFlvKf,7DlZH8EiVDgQXNoj3dnyZC,7lj9PG3guq2A4VLRBLkfKI,4UySkSnMBKf1PS32agnwxp,2PaQb8AtQqIxawPA2O7Vaw,1G2ipJnWC0v0rSPPpzjoek,55Tf67DiltMLjOY6A25s3J,4Ie43xzNjvedClTOHYYp19,6iF4RgIjDvD

SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=3azJifCSqg9fRij2yKIbWz,5NijSs5dAwaIybq1GaRTIe,7MXVkk9YMctZqd1Srtv4MB,65tIJClJx8fHo6YW4wVDhi,5xOAQ5o5mPI2FZZEOu1NZL,7x9aauaA9cu6tyfpHnqDLo,56y1jOTK0XSvJzVv9vHQBK,3rUGC1vUpkDG9CZFHMur1t,01qFKNWq73UfEslI0GvumE,2YSzYUF3jWqb9YP9VXmpjE,1BxfuPKGuaTgP7aM0Bbdwr,5RqSsdzTNPX1uzkmlHCFvK,3vkCueOmm7xQDoJ17W1Pm3,5mjYQaktjmjcMKcUIcqz4s,7ABLbnD53cQK00mhcaOUVG,4bw8mcDUSRWfQo63ZTYRnU,1odExI7RdWc4BT515LTAwj,6uIIdjYTxxpWOyWuVXrKQO,59NraMJsLaMCVtwXTSia8i,4Dvkj6JhhA12EX05fT7y2e,1Qrg8KqiBpW07V7PNxwwwL,3717eMglfGPYr9YKzdsiho,4hv9mEWi4911k1uhU4fPEH,3si1puqOHdvGAFZ7Z4VU27,5yy7ct80WZm20IrCh7KVb6,6hjbdWzbOX4OxBPZUxJVNH,11xjDJ7i9BLd1AYMedwJQK,5sfxl2zZcEdWQOF4ymxGpd,0iqfDCsvFZr9YBwRFqnai6,0ytdPqBPMEHhgXxLbx472A,3ZL94olNtSVGYR1Bmc9acW,3b8UW227uLkRDvd0w9e1Sw,6TQj22qKkIAd9cuBiFlvKf,7DlZH8EiVDgQXNoj3dnyZC,7lj9PG3guq2A4VLRBLkfKI,4UySkSnMBKf1PS32agnwxp,2PaQb8AtQqIxawPA2O7Vaw,1G2ipJnWC0v0rSPPpzjoek,55Tf67DiltMLjOY6A25s3J,4Ie43xzNj